# Tiền xử lý Agoda theo Le et al. cho bài toán LDA

Notebook đọc **duy nhất** `data/agoda_vi_reviews.csv` và ghi **duy nhất** `data/agoda_vi_reviews_preprocessed.csv`. File raw không bị sửa. Quy trình gồm làm sạch, kiểm tra ngôn ngữ từ nội dung, mở rộng viết tắt, thêm dấu có kiểm soát, gán nhãn từ loại và tạo từ đầu vào LDA. Ngưỡng nhóm NPS là Detractors ≤6,4; Passives 6,5–8,2; Promoters ≥8,3. Mã xử lý nằm trong `scripts/preprocess_agoda_reviews.py` để chạy cả từ notebook lẫn dòng lệnh; từng cell dưới đây gọi một bước và kiểm tra kết quả.

Cài phụ thuộc một lần: `python3 -m venv .venv` rồi `.venv/bin/python -m pip install -r requirements-preprocess.txt`. Chọn kernel Python của `.venv` khi mở notebook. Chạy các cell theo thứ tự. Thư viện Underthesea có thể tải model ngôn ngữ ở lần chạy đầu.

## 1. Nạp hàm và cố định đường dẫn

Tìm thư mục dự án từ vị trí notebook. `sha256` chụp dấu vân tay của CSV raw để đối chiếu cuối quy trình. Dữ liệu nguồn luôn được mở ở chế độ đọc.

In [1]:
from pathlib import Path
import sys
from collections import Counter

# Chạy được khi kernel mở ở thư mục gốc hoặc trong notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/agoda_vi_reviews.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy data/agoda_vi_reviews.csv")
sys.path.insert(0, str(ROOT / "scripts"))
from preprocess_agoda_reviews import (
    RAW_PATH, OUTPUT_PATH, sha256, read_raw, clean_text,
    build_accent_lexicon, normalize_abbreviations, classify_language,
    restore_accents, topic_tokens, process_rows, assign_splits, write_and_verify,
)
raw_hash_before = sha256(RAW_PATH)
print("Raw:", RAW_PATH)
print("Output:", OUTPUT_PATH)
print("SHA-256 raw:", raw_hash_before)


Raw: /home/tvphuoc/NLP/data/agoda_vi_reviews.csv
Output: /home/tvphuoc/NLP/data/agoda_vi_reviews_preprocessed.csv
SHA-256 raw: c0a184d01cacb88e49d3f9f2fbe420b06bb10453c98a4c18805c3b16da2aa79b


## 2. Đọc CSV raw và kiểm tra cấu trúc

`read_raw` kiểm tra các cột bắt buộc rồi đọc bằng `csv.DictReader`, giữ đúng dấu phẩy và xuống dòng bên trong review. Cell đếm ô trống và khóa ID trùng để biết chất lượng đầu vào trước khi lọc.

In [2]:
raw_rows, source_columns = read_raw()
raw_keys = [((r.get("hotel_id") or "").strip(),
             (r.get("review_id") or "").strip()) for r in raw_rows]
print("Số dòng raw:", len(raw_rows))
print("Cột:", source_columns)
print("Review body trống:", sum(not (r.get("review_text") or "").strip() for r in raw_rows))
print("Khóa hotel_id + review_id trùng:", len(raw_keys) - len(set(raw_keys)))


Số dòng raw: 100090
Cột: ['hotel_id', 'hotel_name', 'hotel_url', 'review_id', 'review_date', 'rating_10', 'review_title', 'review_text', 'language_code', 'nps_group', 'first_seen_sort', 'first_seen_page']
Review body trống: 0
Khóa hotel_id + review_id trùng: 0


## 3. Làm sạch cơ bản và học từ điển thêm dấu

`clean_text` chuẩn hóa NFC, giải mã HTML, bỏ thẻ và ký tự vô hình, gộp khoảng trắng, thay email/URL/chuỗi giống số điện thoại bằng placeholder. Không xóa dấu tiếng Việt hay từ phủ định. `build_accent_lexicon` học các dạng có dấu từ **chính raw**, nhưng chỉ nhận từ xuất hiện ít nhất 10 lần và một cách viết chiếm ≥92%; trường hợp đa nghĩa được bỏ qua. Từ điển này chỉ tồn tại trong bộ nhớ, không tạo dataset thứ hai.

In [3]:
# Ví dụ làm sạch trước khi áp dụng trên toàn bộ tập.
example = "  Phòng&nbsp;sạch <br> nhân viên tốt!  "
print("Ví dụ:", clean_text(example))
lexicon = build_accent_lexicon(raw_rows)
print("Số mục từ có dấu tin cậy cao:", len(lexicon))
print("Một số mục:", {k: lexicon.get(k) for k in
      ("phong", "sach", "dep", "nhan", "vien", "tot")})


Ví dụ: Phòng sạch nhân viên tốt!
Số mục từ có dấu tin cậy cao: 8
Một số mục: {'phong': 'phòng', 'sach': 'sạch', 'dep': None, 'nhan': None, 'vien': None, 'tot': 'tốt'}


## 4. Kiểm tra ngôn ngữ, viết tắt và thêm dấu

Bài báo yêu cầu kiểm tra nội dung thay vì tin tuyệt đối `language_code=vi`. `classify_language` dùng FastText qua Underthesea cho câu đủ dài, kết hợp dấu/cụm đặc trưng tiếng Việt cho câu ngắn hoặc không dấu. Câu Anh và câu quá mơ hồ bị loại. Từ điển viết tắt chỉ mở rộng các dạng rõ nghĩa trong review khách sạn (`ks`, `nv`, `ko`, `dc`, ...). Nếu câu gốc hoàn toàn không dấu, `restore_accents` thử cụm miền du lịch và từ đơn nghĩa trong từ điển; kết quả chưa chắc được giữ nguyên và gắn `accent_status`. **Không coi bước này là khôi phục dấu hoàn hảo.**

In [4]:
examples = [
    "Phòng sạch đẹp, nhân viên thân thiện.",
    "phong sach dep nhan vien than thien",
    "Nice Hotel, nice staff and great service.",
    "Tốt",
]
for original in examples:
    body = clean_text(original)
    detected, rule = classify_language(body)
    expanded, _ = normalize_abbreviations(body)
    restored, status = restore_accents(expanded, lexicon,
                                      original_has_accents=any(c in body.lower() for c in "đăâêôơưàáạảãèéẹẻẽìíịỉĩòóọỏõùúụủũỳýỵỷỹ"))
    print(repr(original), "=>", detected, rule, "|", restored, status)


'Phòng sạch đẹp, nhân viên thân thiện.' => vi fasttext | phòng sạch đẹp, nhân viên thân thiện. already_accented
'phong sach dep nhan vien than thien' => vi vietnamese_cues | phòng sạch dep nhân viên thân thiện partially_restored
'Nice Hotel, nice staff and great service.' => en fasttext_other | nice hotel, nice staff and great service. unresolved_unaccented
'Tốt' => vi short_with_accent | tốt unresolved_unaccented


## 5. Tách từ, gán POS và tạo văn bản LDA

Underthesea `pos_tag` vừa tách từ tiếng Việt vừa gán từ loại. Hàm `topic_tokens` giữ nhãn bắt đầu bằng `N` (danh từ) hoặc `A` (tính từ), bỏ từ dừng và ký hiệu, nối từ ghép bằng `_`. Ví dụ `nhân viên` thành `nhân_viên`. Chỉ **thân review** đi vào LDA; tiêu đề lưu riêng để tham khảo. Sau POS, review không còn từ chủ đề bị loại. Chạy toàn bộ 100.090 dòng có thể mất vài phút.

In [5]:
print("Ví dụ POS/LDA:", topic_tokens("phòng sạch đẹp, nhân viên thân thiện"))
processed_rows, audit = process_rows(raw_rows, lexicon)
print("Số dòng được giữ:", len(processed_rows))
print("Kiểm toán lọc/xử lý:", dict(audit))
print("Mẫu LDA:", processed_rows[0]["lda_text"] if processed_rows else "(rỗng)")


Ví dụ POS/LDA: ['phòng', 'sạch_đẹp', 'nhân_viên']
Đã xử lý 10,000/100,090 review gốc
Đã xử lý 20,000/100,090 review gốc
Đã xử lý 30,000/100,090 review gốc
Đã xử lý 40,000/100,090 review gốc
Đã xử lý 50,000/100,090 review gốc
Đã xử lý 60,000/100,090 review gốc
Đã xử lý 70,000/100,090 review gốc
Đã xử lý 80,000/100,090 review gốc
Đã xử lý 90,000/100,090 review gốc
Đã xử lý 100,000/100,090 review gốc
Số dòng được giữ: 87837
Kiểm toán lọc/xử lý: {'html_tags': 1, 'invisible_chars': 18, 'emails': 4, 'urls': 4, 'phone_candidates': 23, 'abbreviations_expanded': 30911, 'dropped_language_en': 4795, 'dropped_language_unknown': 4986, 'reviews_partially_restored': 514, 'dropped_language_it': 19, 'dropped_no_topic_tokens': 2251, 'dropped_language_fr': 44, 'reviews_unresolved_unaccented': 83, 'dropped_language_pl': 4, 'dropped_language_hr': 5, 'dropped_language_kv': 1, 'dropped_language_gd': 1, 'dropped_language_io': 1, 'dropped_language_ja': 6, 'dropped_language_ca': 11, 'dropped_language_jv': 3, 'd

## 6. Giữ nhóm NPS và split cũ để truy vết

Nhãn tính lại từ `rating_10` theo đúng ngưỡng bài báo. `assign_splits` giữ cột train/validation/test theo khách sạn để tương thích những thí nghiệm supervised trước; **LDA của bài báo không dùng cột split** và được chạy riêng cho từng nhóm NPS. Không đưa điểm số hoặc nhãn vào văn bản LDA.

In [6]:
split_counts = assign_splits(processed_rows)
label_counts = Counter(row["label"] for row in processed_rows)
print("NPS:", dict(label_counts))
print("Split:", dict(split_counts))
print("Số khách sạn:", len({row["hotel_id"] for row in processed_rows}))


NPS: {'Promoters': 57255, 'Passives': 12462, 'Detractors': 18120}
Split: {'train': 70959, 'validation': 7761, 'test': 9117}
Số khách sạn: 2198


## 7. Ghi **một CSV đầu ra** và đối chiếu

Ghi vào file tạm rồi thay thế đúng đường dẫn đầu ra hiện có. Hàm kiểm tra không có ID trùng, không có `lda_text` rỗng, mọi dòng giữ lại đều xác định tiếng Việt, một khách sạn thuộc một split, và SHA-256 raw không đổi. Không tạo bản CSV phụ.

In [7]:
raw_hash_after, saved_labels = write_and_verify(processed_rows, raw_hash_before)
print("Đầu ra:", OUTPUT_PATH)
print("Số dòng:", len(processed_rows))
print("Nhãn đã lưu:", dict(saved_labels))
print("SHA-256 raw không đổi:", raw_hash_after)


Đầu ra: /home/tvphuoc/NLP/data/agoda_vi_reviews_preprocessed.csv
Số dòng: 87837
Nhãn đã lưu: {'Promoters': 57255, 'Passives': 12462, 'Detractors': 18120}
SHA-256 raw không đổi: c0a184d01cacb88e49d3f9f2fbe420b06bb10453c98a4c18805c3b16da2aa79b


## 8. Cách dùng dữ liệu cho bài báo

CSV đầu ra chứa `review_text_clean` (sạch cơ bản), `review_text_normalized` (mở rộng viết tắt/thêm dấu có kiểm soát) và `lda_text` (danh từ/tính từ đã tách từ, cách nhau bằng dấu cách). Dùng **`lda_text`** để tạo từ điển và corpus LDA, tách theo `label`: Detractors, Passives, Promoters. Sau đó thử số topic K=1…6, chọn theo coherence và diễn giải từ khóa. Các cột `rating_10`, `label`, `split`, ID và thời gian chỉ là metadata. `accent_status` và `language_rule` giúp rà soát các quyết định tự động. Phương pháp đã bám quy trình bài báo, nhưng công cụ ngôn ngữ và danh sách viết tắt/từ dừng là lựa chọn triển khai của dự án, vì bài báo không công bố cấu hình chi tiết.